# 03. 마스킹 복원과 멀티모달 결합 심화 실습

목표: S2Vec의 MAE 아이디어를 작은 선형 복원 문제로 단순화하고, 건조 환경 임베딩과 이미지형 임베딩을 결합하는 효과를 실험합니다.

실행 방법:

```bash
python -m pip install -r s2vec-geospatial-embeddings/requirements.txt
```

주의: 이 노트북의 복원기는 실제 Vision Transformer가 아닙니다. 논문의 핵심 학습 신호를 이해하기 위한 축소 모형입니다.

## 1. 합성 데이터 생성

S2Vec가 잘 맞는 사회경제 타깃과, 위성 이미지 피처가 더 직접적으로 맞는 환경 타깃을 동시에 만듭니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

N_PARENTS = 180
GRID = 16
FEATURES = ['food', 'retail', 'office', 'transit', 'roads', 'parks', 'housing', 'industrial']
F = len(FEATURES)
idx = {name: i for i, name in enumerate(FEATURES)}

urban_profile = np.array([8, 7, 9, 6, 7, 1, 5, 1], dtype=float)
suburb_profile = np.array([3, 4, 2, 2, 5, 4, 8, 1], dtype=float)
green_profile = np.array([1, 1, 0.2, 0.3, 1, 12, 1, 0.2], dtype=float)
industrial_profile = np.array([1, 1, 2, 1, 5, 0.5, 1, 8], dtype=float)

def minmax(values):
    values = np.asarray(values, dtype=float)
    return (values - values.min()) / (values.max() - values.min() + 1e-9)

def make_parent_cell(parent_x, parent_y):
    cube = np.zeros((GRID, GRID, F), dtype=float)
    metro_strength = np.exp(-((parent_x - 0.40) ** 2 + (parent_y - 0.55) ** 2) / 0.07)
    green_strength = np.exp(-((parent_x - 0.82) ** 2 + (parent_y - 0.30) ** 2) / 0.04)
    industrial_strength = np.exp(-((parent_x - 0.24) ** 2 + (parent_y - 0.22) ** 2) / 0.05)
    for r in range(GRID):
        for c in range(GRID):
            lx = c / (GRID - 1)
            ly = r / (GRID - 1)
            center = np.exp(-((lx - 0.5) ** 2 + (ly - 0.5) ** 2) / 0.055)
            park = np.exp(-((lx - 0.15) ** 2 + (ly - 0.80) ** 2) / 0.045)
            industry = np.exp(-((ly - 0.18) ** 2) / 0.018)
            lam = 0.20
            lam += (0.35 + metro_strength) * center * urban_profile
            lam += (0.55 - 0.20 * metro_strength) * suburb_profile
            lam += (0.20 + green_strength) * park * green_profile
            lam += (0.10 + industrial_strength) * industry * industrial_profile
            cube[r, c] = rng.poisson(np.clip(lam, 0.05, None))
    return cube

parent_xy = rng.random((N_PARENTS, 2))
cubes = np.stack([make_parent_cell(x, y) for x, y in parent_xy])
print('built-environment tensor:', cubes.shape)

## 2. 마스킹 복원용 학습 데이터

진짜 MAE는 보이는 패치들을 Transformer encoder에 넣고, decoder가 가려진 패치를 복원합니다. 여기서는 가려진 셀 하나를 복원하기 위해 `주변 보이는 셀 평균 + 부모 셀 평균 + 셀 좌표`를 입력으로 쓰는 선형 복원기를 학습합니다.

In [ ]:
def descriptor_for_patch(cube, visible_mask, row, col):
    r0, r1 = max(0, row - 1), min(GRID, row + 2)
    c0, c1 = max(0, col - 1), min(GRID, col + 2)
    local_visible = visible_mask[r0:r1, c0:c1]
    local_values = cube[r0:r1, c0:c1, :][local_visible]
    parent_values = cube[visible_mask]

    # 주변이 모두 가려진 드문 경우에는 부모 셀 평균을 사용합니다.
    parent_mean = parent_values.mean(axis=0) if len(parent_values) else cube.reshape(-1, F).mean(axis=0)
    local_mean = local_values.mean(axis=0) if len(local_values) else parent_mean
    coord = np.array([row / (GRID - 1), col / (GRID - 1)], dtype=float)
    return np.concatenate([local_mean, parent_mean, coord])

def build_masked_reconstruction_dataset(cubes, masks_per_parent=2, max_masked_per_parent=24, mask_ratio=0.50):
    X, Y = [], []
    for cube in cubes:
        for _ in range(masks_per_parent):
            masked = rng.random((GRID, GRID)) < mask_ratio
            visible = ~masked
            masked_positions = np.argwhere(masked)
            rng.shuffle(masked_positions)
            for row, col in masked_positions[:max_masked_per_parent]:
                X.append(descriptor_for_patch(cube, visible, int(row), int(col)))
                Y.append(cube[row, col])
    return np.asarray(X), np.asarray(Y)

X_recon, Y_recon = build_masked_reconstruction_dataset(cubes)
print('reconstruction X:', X_recon.shape)
print('reconstruction target:', Y_recon.shape)

## 3. 선형 복원기 학습

여기서는 multi-output ridge regression을 사용합니다. 복원 성능을 전역 평균 baseline과 비교하면, 주변 문맥이 원래 피처를 얼마나 설명하는지 볼 수 있습니다.

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-6
    return (X_train - mean) / std, (X_test - mean) / std, mean, std

def fit_multi_ridge(X, Y, alpha=5.0):
    Xb = np.column_stack([np.ones(len(X)), X])
    penalty = np.eye(Xb.shape[1]) * alpha
    penalty[0, 0] = 0.0
    return np.linalg.solve(Xb.T @ Xb + penalty, Xb.T @ Y)

def predict_multi_ridge(X, W):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ W

order = rng.permutation(len(X_recon))
cut = int(0.80 * len(order))
train_idx, test_idx = order[:cut], order[cut:]

X_train, X_test, recon_mean, recon_std = standardize_train_test(X_recon[train_idx], X_recon[test_idx])
W_recon = fit_multi_ridge(X_train, Y_recon[train_idx])
pred_test = predict_multi_ridge(X_test, W_recon)

global_baseline = np.repeat(Y_recon[train_idx].mean(axis=0, keepdims=True), len(test_idx), axis=0)
model_mae = np.mean(np.abs(pred_test - Y_recon[test_idx]))
baseline_mae = np.mean(np.abs(global_baseline - Y_recon[test_idx]))

print('global mean baseline MAE:', round(float(baseline_mae), 3))
print('context reconstruction MAE:', round(float(model_mae), 3))

## 4. 복원기를 임베딩 생성기로 사용하기

S2Vec는 학습된 encoder의 patch 출력을 임베딩으로 사용합니다. 이 축소 실험에서는 `원래 카운트 + 문맥으로 예측한 카운트`를 S2식 임베딩으로 둡니다. 복원 예측값은 주변 맥락에서 기대되는 지역 특성을 담습니다.

In [ ]:
def mean_pool_3x3(batch):
    padded = np.pad(batch, ((0, 0), (1, 1), (1, 1), (0, 0)), mode='edge')
    pooled = np.zeros_like(batch, dtype=float)
    for dr in range(3):
        for dc in range(3):
            pooled += padded[:, dr:dr + GRID, dc:dc + GRID, :]
    return pooled / 9.0

raw_X = cubes.reshape(-1, F)
local_context = mean_pool_3x3(cubes).reshape(-1, F)
parent_context = np.repeat(cubes.mean(axis=(1, 2)), GRID * GRID, axis=0)
patch_xy_one = np.array([(r / (GRID - 1), c / (GRID - 1)) for r in range(GRID) for c in range(GRID)])
patch_xy = np.tile(patch_xy_one, (N_PARENTS, 1))
geo_xy = np.repeat(parent_xy, GRID * GRID, axis=0)

descriptor_all = np.concatenate([local_context, parent_context, patch_xy], axis=1)
descriptor_all_std = (descriptor_all - recon_mean) / recon_std
reconstructed_expected_counts = predict_multi_ridge(descriptor_all_std, W_recon)

s2_embedding = np.concatenate([raw_X, reconstructed_expected_counts, geo_xy], axis=1)
print('S2-like embedding shape:', s2_embedding.shape)

## 5. 이미지형 모달리티와 타깃 만들기

논문에서 환경 과제는 위성 이미지 기반 임베딩이 더 강했습니다. 이를 흉내 내기 위해 vegetation/elevation 신호를 가진 이미지형 임베딩을 따로 만듭니다.

In [ ]:
vegetation = minmax(0.55 * raw_X[:, idx['parks']] + 0.35 * geo_xy[:, 1] + rng.normal(0, 0.8, len(raw_X)))
elevation = minmax(0.70 * geo_xy[:, 0] + 0.20 * np.sin(6 * geo_xy[:, 1]) + rng.normal(0, 0.15, len(raw_X)))
texture = minmax(0.30 * raw_X[:, idx['roads']] + 0.30 * raw_X[:, idx['industrial']] + rng.normal(0, 1.0, len(raw_X)))
image_embedding = np.column_stack([vegetation, elevation, texture])

socioeconomic_target = minmax(
    0.22 * s2_embedding[:, idx['office']]
    + 0.18 * s2_embedding[:, idx['transit']]
    + 0.14 * s2_embedding[:, idx['retail']]
    + 0.16 * geo_xy[:, 0]
    - 0.10 * raw_X[:, idx['industrial']]
    + rng.normal(0, 0.8, len(raw_X))
)

environment_target = minmax(0.75 * vegetation + 0.25 * elevation + rng.normal(0, 0.08, len(raw_X)))

print('image embedding shape:', image_embedding.shape)
print('targets:', socioeconomic_target.shape, environment_target.shape)

## 6. 단일 모달과 결합 모달 비교

아래 비교는 논문 결과의 방향성을 축소해서 보여줍니다. 사회경제 타깃은 S2식 피처가 강하고, 환경 타깃은 이미지형 피처가 강해야 정상입니다. 결합은 두 모달리티가 서로 다른 정보를 가질 때 유리합니다.

In [ ]:
def random_projection(X, out_dim, seed):
    local_rng = np.random.default_rng(seed)
    W = local_rng.normal(0, 1 / np.sqrt(X.shape[1]), size=(X.shape[1], out_dim))
    return X @ W

concat_embedding = np.concatenate([s2_embedding, image_embedding], axis=1)
proj_add_embedding = random_projection(s2_embedding, 24, 1) + random_projection(image_embedding, 24, 2)

feature_sets = [
    ('S2-like only', s2_embedding),
    ('image-like only', image_embedding),
    ('concat fusion', concat_embedding),
    ('project-add fusion', proj_add_embedding),
]

geo_test = geo_xy[:, 0] > 0.72
geo_train = ~geo_test

def standardize_by_train(X, train_mask):
    mean = X[train_mask].mean(axis=0)
    std = X[train_mask].std(axis=0) + 1e-6
    return (X - mean) / std

def fit_ridge(X, y, alpha=4.0):
    Xb = np.column_stack([np.ones(len(X)), X])
    penalty = np.eye(Xb.shape[1]) * alpha
    penalty[0, 0] = 0.0
    return np.linalg.solve(Xb.T @ Xb + penalty, Xb.T @ y)

def predict_ridge(X, weights):
    return np.column_stack([np.ones(len(X)), X]) @ weights

def r2_score(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / (np.sum((y_true - y_true.mean()) ** 2) + 1e-12)

def evaluate_feature_set(X, y):
    Xs = standardize_by_train(X, geo_train)
    weights = fit_ridge(Xs[geo_train], y[geo_train])
    pred = predict_ridge(Xs[geo_test], weights)
    return r2_score(y[geo_test], pred)

rows = []
for name, X in feature_sets:
    rows.append((name, evaluate_feature_set(X, socioeconomic_target), evaluate_feature_set(X, environment_target)))

print('{:<20} {:>12} {:>12}'.format('features', 'socio R2', 'env R2'))
print('-' * 48)
for name, socio_r2, env_r2 in rows:
    print('{:<20} {:>12.3f} {:>12.3f}'.format(name, socio_r2, env_r2))

In [ ]:
labels = [row[0] for row in rows]
socio_scores = [row[1] for row in rows]
env_scores = [row[2] for row in rows]
x = np.arange(len(labels))

plt.figure(figsize=(10, 4))
plt.bar(x - 0.18, socio_scores, width=0.36, label='socioeconomic')
plt.bar(x + 0.18, env_scores, width=0.36, label='environment')
plt.xticks(x, labels, rotation=15, ha='right')
plt.ylabel('R2 on geographic holdout')
plt.title('modality comparison on synthetic tasks')
plt.legend()
plt.tight_layout()
plt.show()

## 해석과 확장 과제

- 마스킹 복원기는 주변 맥락만으로 숨겨진 피처를 어느 정도 예측할 수 있습니다. 이것이 S2Vec의 자기지도 학습 신호입니다.
- 사회경제 타깃은 상업, 교통, 주거 피처와 강하게 연결되므로 S2식 임베딩이 유리합니다.
- 환경 타깃은 vegetation/elevation 같은 이미지형 신호가 직접적이므로 이미지 모달리티가 유리합니다.
- 결합 모델은 두 모달리티가 서로 다른 정보를 담을 때 유리하지만, 단일 모달이 이미 충분히 강하면 항상 이긴다고 보장할 수 없습니다.

확장 실험: `mask_ratio`, 부모 셀 크기, 지리적 holdout 기준, fusion 방식을 바꿔 결과가 얼마나 민감하게 달라지는지 확인해 보세요.